# Campus Route Finder Using Breadth-First Search (BFS)

## Project Overview & Objectives
This project implements a smart navigational assistant for a college campus using the **Breadth-First Search (BFS)** algorithm on an unweighted graph.

### Objectives:
1. **Graph Representation:** Model campus buildings and interconnected walkways as an undirected graph.
2. **Optimal Routing:** Find the path between any selected source and destination with the minimum number of hops (pathway connections).
3. **Step-by-Step Directions:** Provide clear, human-readable turn-by-turn walking instructions with illustrative distance and time estimates.
4. **Visual Mapping:** Render an intuitive map displaying the campus layout, color-coded route nodes, and a descriptive legend.
5. **Academic Trace:** Expose the internal BFS queue state at each step to demonstrate the algorithm during a college viva.

---
### Why Breadth-First Search (BFS)?
In an unweighted graph where every pathway connection represents 1 hop:
- BFS explores nodes level by level using a **FIFO (First-In, First-Out) Queue**.
- BFS is mathematically guaranteed to find the path with the fewest hops between any two nodes.
- Unlike Depth-First Search (DFS), BFS does not wander down deep paths before finding the shortest route.

## 1. Environment Setup & Library Imports
We import standard Python libraries (`collections.deque`, `matplotlib.pyplot`) and provide graceful handling for `networkx` and `ipywidgets`.

In [ ]:
# Step 1: Library Imports & Environment Configuration
from collections import deque
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Check NetworkX availability (pre-installed in Google Colab)
try:
    import networkx as nx
    HAS_NETWORKX = True
except ImportError:
    HAS_NETWORKX = False
    print("[Notice] NetworkX not detected locally. Matplotlib native rendering fallback enabled.")

# Check ipywidgets availability (for interactive dropdowns in Jupyter / Google Colab)
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("[Notice] ipywidgets not detected. Using clean console / programmatic input fallback.")


## 2. Campus Graph & Coordinate Layout
We represent the 7 verified campus locations and their physical walkway connections as an adjacency list dictionary.

> **Note on Distances:** In an unweighted graph, BFS strictly optimizes the number of hops. The segment distances (meters) and walking times (minutes) below are realistic illustrative estimates provided for practical navigational guidance.

In [ ]:
# Step 2: Campus Graph Definition & Metadata

# Verified campus locations and unweighted walkway connections
campus_graph = {
    "Main Gate": ["Library", "Canteen"],
    "Library": ["Main Gate", "Admin Block", "Computer Lab"],
    "Canteen": ["Main Gate", "Auditorium"],
    "Admin Block": ["Library", "Computer Lab", "Classroom"],
    "Computer Lab": ["Library", "Admin Block", "Classroom"],
    "Auditorium": ["Canteen", "Classroom"],
    "Classroom": ["Admin Block", "Computer Lab", "Auditorium"]
}

# 2D Map Coordinates (X, Y) for layout visualization
campus_positions = {
    "Main Gate": (0.0, 2.0),
    "Library": (-2.0, 1.0),
    "Canteen": (2.0, 1.0),
    "Admin Block": (-2.0, 0.0),
    "Computer Lab": (0.0, 0.0),
    "Auditorium": (2.0, 0.0),
    "Classroom": (0.0, -1.0)
}

# Illustrative walkway measurements: (approx_meters, approx_walking_minutes)
sample_walkway_estimates = {
    tuple(sorted(["Main Gate", "Library"])): (120, 2),
    tuple(sorted(["Main Gate", "Canteen"])): (150, 2.5),
    tuple(sorted(["Library", "Admin Block"])): (80, 1),
    tuple(sorted(["Library", "Computer Lab"])): (100, 1.5),
    tuple(sorted(["Canteen", "Auditorium"])): (110, 1.5),
    tuple(sorted(["Admin Block", "Computer Lab"])): (90, 1),
    tuple(sorted(["Admin Block", "Classroom"])): (130, 2),
    tuple(sorted(["Computer Lab", "Classroom"])): (85, 1),
    tuple(sorted(["Auditorium", "Classroom"])): (140, 2)
}

print(f"Total Campus Locations: {len(campus_graph)}")
print("Available Campus Buildings:")
for idx, (loc, neighbors) in enumerate(campus_graph.items(), start=1):
    print(f"  {idx}. {loc:14} -> Direct Walkways to: {', '.join(neighbors)}")


## 3. Breadth-First Search (BFS) Algorithm with Viva Trace
The function below implements BFS using `collections.deque`.
- **FIFO Queue:** Explores paths in order of length (length 1, then length 2, then length 3...).
- **Visited Set:** Avoids revisiting nodes and prevents infinite loops in cyclical graphs.
- **Trace Recorder:** Logs each queue step so evaluators during a college viva can observe the traversal.

In [ ]:
# Step 3: BFS Implementation with Viva Traversal Log

def bfs_shortest_path(graph, start, destination):
    """
    Finds the shortest path between start and destination using BFS.
    Returns:
        path (list): Sequence of buildings from start to destination, or None if unreachable.
        trace_log (list): Detailed trace of node explorations for viva demonstration.
    """
    if start not in graph or destination not in graph:
        return None, []

    if start == destination:
        return [start], [{"step": 1, "current_node": start, "current_path": [start], "queue": []}]

    # Queue holds candidate paths: e.g. [[start]]
    queue = deque([[start]])
    visited = {start}
    trace_log = []
    step = 1

    while queue:
        path = queue.popleft()
        current = path[-1]

        # Log current expansion for viva walkthrough
        queue_snapshot = [" -> ".join(p) for p in list(queue)]
        trace_log.append({
            "step": step,
            "current_node": current,
            "current_path": list(path),
            "queue": queue_snapshot
        })
        step += 1

        # When destination is first reached, BFS guarantees this is shortest path in hops
        if current == destination:
            return path, trace_log

        # Expand unvisited neighbors
        for neighbor in graph[current]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(path + [neighbor])

    return None, trace_log


## 4. Human-Readable Step-by-Step Directions Generator
Converts the raw path list into structured, practical walking instructions with estimated distances and walking times.

In [ ]:
# Step 4: Step-by-Step Directions Generator

def generate_directions(path):
    """
    Formats turn-by-turn guidance, hop counts, and illustrative walking metrics.
    """
    if not path:
        return "No route available."
    if len(path) == 1:
        return f"You are already at [{path[0]}]. No movement needed."

    total_dist = 0
    total_time = 0.0
    instructions = []

    instructions.append("=" * 62)
    instructions.append(f"  STEP-BY-STEP DIRECTIONS: {path[0]} -> {path[-1]}")
    instructions.append("=" * 62)

    for i in range(len(path) - 1):
        src = path[i]
        dst = path[i + 1]
        pair = tuple(sorted([src, dst]))
        dist, time_est = sample_walkway_estimates.get(pair, (100, 1.5))
        total_dist += dist
        total_time += time_est

        step_idx = i + 1
        if i == 0:
            desc = f"Step {step_idx}: Depart from [{src}] and walk along pathway to [{dst}]."
        elif i == len(path) - 2:
            desc = f"Step {step_idx}: From [{src}], proceed directly to destination [{dst}]."
        else:
            desc = f"Step {step_idx}: From [{src}], cross through toward [{dst}]."

        instructions.append(f"  {desc}")
        instructions.append(f"    |-- Segment: ~{dist} meters (approx. ~{time_est} min)")

    instructions.append("-" * 62)
    instructions.append(f"  Total Pathway Connections (Hops): {len(path) - 1}")
    instructions.append(f"  Total Campus Locations Visited: {len(path)}")
    instructions.append(f"  Estimated Total Walking Distance: ~{total_dist} meters (approx. ~{total_time:.1f} mins)")
    instructions.append("  *Note: Distances and times are illustrative estimates for demo guidance.")
    instructions.append("=" * 62)

    return "\n".join(instructions)


## 5. Enhanced Campus Map & Route Visualization
Renders a high-contrast visual campus map with:
- **Start Location:** Vivid Green (`#27ae60`)
- **Destination Location:** Bright Red (`#e74c3c`)
- **Route Waypoints:** Amber Orange (`#f39c12`)
- **Other Buildings:** Soft Sky Blue (`#74b9ff`)
- **Shortest Path:** Highlighted bold red line (`width=4.0`)
- **Explanatory Legend:** Clear labels for each color

In [ ]:
# Step 5: Route Map Visualization

def plot_campus_route(graph, positions, path=None, title=None):
    """
    Renders campus map using NetworkX (or native Matplotlib fallback) with a clear color legend.
    """
    fig, ax = plt.subplots(figsize=(11, 6.5), dpi=100)

    start_node = path[0] if path else None
    end_node = path[-1] if path else None
    route_nodes = set(path) if path else set()
    route_edges = set([tuple(sorted([path[i], path[i+1]])) for i in range(len(path)-1)]) if path else set()

    # Color palette
    C_START = "#27ae60"      # Emerald Green
    C_DEST = "#e74c3c"       # Crimson Red
    C_WAYPOINT = "#f39c12"   # Amber Orange
    C_BUILDING = "#74b9ff"   # Sky Blue
    C_EDGE_ROUTE = "#c0392b" # Bold Red
    C_EDGE_BASE = "#b2bec3"  # Slate Gray

    if HAS_NETWORKX:
        G = nx.Graph()
        for u, neighbors in graph.items():
            for v in neighbors:
                G.add_edge(u, v)

        # 1. Base edges
        base_edges = [(u, v) for u, v in G.edges() if tuple(sorted([u, v])) not in route_edges]
        nx.draw_networkx_edges(G, positions, edgelist=base_edges, edge_color=C_EDGE_BASE, width=1.8, ax=ax)

        # 2. Shortest route edges
        if path and len(path) > 1:
            active_edges = [(path[i], path[i+1]) for i in range(len(path)-1)]
            nx.draw_networkx_edges(G, positions, edgelist=active_edges, edge_color=C_EDGE_ROUTE, width=4.0, ax=ax)

        # 3. Node color assignment
        node_colors = []
        for node in G.nodes():
            if node == start_node:
                node_colors.append(C_START)
            elif node == end_node:
                node_colors.append(C_DEST)
            elif node in route_nodes:
                node_colors.append(C_WAYPOINT)
            else:
                node_colors.append(C_BUILDING)

        nx.draw_networkx_nodes(G, positions, node_color=node_colors, node_size=3200, edgecolors="#2d3436", linewidths=1.5, ax=ax)
        nx.draw_networkx_labels(G, positions, font_size=9, font_weight="bold", font_color="#2d3436", ax=ax)

    else:
        # Native Matplotlib rendering fallback
        drawn_edges = set()
        for u, neighbors in graph.items():
            for v in neighbors:
                pair = tuple(sorted([u, v]))
                if pair not in drawn_edges:
                    drawn_edges.add(pair)
                    xs = [positions[u][0], positions[v][0]]
                    ys = [positions[u][1], positions[v][1]]
                    if pair in route_edges:
                        ax.plot(xs, ys, color=C_EDGE_ROUTE, linewidth=4.0, zorder=2)
                    else:
                        ax.plot(xs, ys, color=C_EDGE_BASE, linewidth=1.8, linestyle="--", zorder=1)

        for node, (x, y) in positions.items():
            if node == start_node:
                color = C_START
            elif node == end_node:
                color = C_DEST
            elif node in route_nodes:
                color = C_WAYPOINT
            else:
                color = C_BUILDING

            circle = plt.Circle((x, y), 0.32, facecolor=color, edgecolor="#2d3436", linewidth=1.5, zorder=3)
            ax.add_patch(circle)
            ax.text(x, y, node, fontsize=9, fontweight="bold", color="#2d3436", ha="center", va="center", zorder=4)

        ax.set_xlim(-3.0, 3.0)
        ax.set_ylim(-2.0, 3.0)

    # Custom Legend
    legend_elements = [
        mpatches.Patch(color=C_START, label="Start Location"),
        mpatches.Patch(color=C_DEST, label="Destination Location"),
        mpatches.Patch(color=C_WAYPOINT, label="Waypoints on Route"),
        mpatches.Patch(color=C_BUILDING, label="Other Campus Buildings"),
        plt.Line2D([0], [0], color=C_EDGE_ROUTE, lw=3.5, label="Shortest Route (BFS)")
    ]
    ax.legend(handles=legend_elements, loc="upper right", frameon=True, facecolor="white", edgecolor="#b2bec3", fontsize=8.5)

    header = title if title else (f"Campus Route: {start_node} -> {end_node}" if path else "Campus Map Layout")
    ax.set_title(header, fontsize=13, fontweight="bold", pad=15)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


## 6. Execution & Interactive Navigation Controller
You can select the starting point and destination:
- **In Google Colab / Jupyter:** Use the interactive dropdown selectors and click **Find Shortest Route**.
- **In standard Python console:** The function runs sample queries or accepts numbered terminal inputs (`1` to `7`).

In [ ]:
# Step 6: Navigation Controller & Interactive Interface

def find_and_display_route(start_loc, dest_loc, show_viva_trace=True):
    """
    Executes BFS route search, prints directions & viva trace, and plots the map.
    """
    path, trace = bfs_shortest_path(campus_graph, start_loc, dest_loc)

    if not path:
        print(f"\nNo route exists between [{start_loc}] and [{dest_loc}].")
        return

    # 1. Output Route Summary
    print("\n" + "=" * 62)
    print(f"  SHORTEST ROUTE (BFS): {' -> '.join(path)}")
    print(f"  Number of Hops: {len(path) - 1} | Locations Visited: {len(path)}")
    print("=" * 62 + "\n")

    # 2. Output Step-by-Step Directions
    print(generate_directions(path))
    print()

    # 3. Output Viva Traversal Trace (if requested)
    if show_viva_trace:
        print("-" * 62)
        print("  COLLEGE VIVA TRACE: BFS QUEUE EXPLORATION SEQUENCE")
        print("-" * 62)
        for rec in trace:
            step_i = rec["step"]
            node = rec["current_node"]
            path_str = " -> ".join(rec["current_path"])
            print(f"  Step {step_i}: Expanded [{node}] | Path so far: {path_str}")
        print("-" * 62 + "\n")

    # 4. Render Enhanced Visual Route Map
    plot_campus_route(campus_graph, campus_positions, path)


# Location list for dropdown widgets
available_locations = list(campus_graph.keys())

if HAS_WIDGETS:
    start_selector = widgets.Dropdown(options=available_locations, value="Main Gate", description="Start:")
    dest_selector = widgets.Dropdown(options=available_locations, value="Classroom", description="Destination:")
    viva_checkbox = widgets.Checkbox(value=True, description="Show Viva Trace")
    search_button = widgets.Button(description="Find Shortest Route", button_style="success", icon="search")
    output_panel = widgets.Output()

    def handle_search(btn):
        with output_panel:
            clear_output(wait=True)
            find_and_display_route(start_selector.value, dest_selector.value, viva_checkbox.value)

    search_button.on_click(handle_search)

    print("Select your source and destination buildings below:")
    display(widgets.VBox([start_selector, dest_selector, viva_checkbox, search_button, output_panel]))

    # Display initial route calculation
    with output_panel:
        find_and_display_route("Main Gate", "Classroom", show_viva_trace=True)

else:
    # Clean programmatic execution for standard terminal / non-widget environments
    print("Executing Demonstration Search: Main Gate -> Classroom")
    find_and_display_route("Main Gate", "Classroom", show_viva_trace=True)


## 7. Algorithm Complexity Analysis & Viva Q&A Reference

### Time & Space Complexity
| Metric | Complexity | Explanation |
| :--- | :---: | :--- |
| **Time Complexity** | $\mathcal{O}(V + E)$ | $V = 7$ vertices (buildings) and $E = 9$ edges (walkways). Every building is enqueued at most once, and every connection is inspected once. |
| **Space Complexity** | $\mathcal{O}(V)$ | The queue and the `visited` set store at most $V$ campus locations simultaneously. |

---

### Viva Q&A Points for Demonstration
1. **Why does BFS guarantee the shortest route?**
   - Because BFS expands nodes level by level. It checks all paths of 1 hop, then 2 hops, then 3 hops, etc. The first time the destination is dequeued, it is guaranteed to have the minimal number of hops.
2. **Why use `collections.deque` instead of a Python `list`?**
   - A Python `list.pop(0)` takes $\mathcal{O}(n)$ time because all subsequent elements must be shifted in memory. `deque.popleft()` takes $\mathcal{O}(1)$ constant time, ensuring optimal performance.
3. **When would you use Dijkstra's Algorithm instead of BFS?**
   - If walkways had significantly unequal lengths and we needed to minimize exact distance in meters rather than hop counts, Dijkstra's algorithm would be preferred. For unweighted graphs (all hops equal cost), BFS is both simpler and faster ($\mathcal{O}(V + E)$ vs $\mathcal{O}((V + E) \log V)$).